# Train DeBERTa-v3-base (v3-3) on Colab or locally

Project B - Fox-vs-NBC headline classifier. This is the **third** training notebook in the
deberta-v3 series. Output:

- `submission/model.pt`  (~735 MB, single-model state_dict)
- `submission/model.py`  (single-model wrapper)
- `submission/preprocess.py`  (Fox=1, NBC=0)
- `submission.zip`  (all three, ready to download)

## What's different in v3-3

- **Data**: trained on `url_with_headlines.csv` (the smaller, freshly scraped slice — ~3.8k rows)
  rather than the 14.5k row `url_with_headlines_20k.csv`. The smaller corpus makes more headroom
  for the hidden-test distribution; FGM + R-Drop regularization are kept to fight the higher
  overfitting risk that comes with less data.
- Hyperparameters mirror v3.2: AdamW + LLRD, R-Drop (α=1.2), label smoothing 0.1,
  weight decay 0.02, FGM (ε=1.0) on word_embeddings, AEDA augmentation on the train fold.

## Prerequisites

1. **GPU runtime** - `Runtime -> Change runtime type -> T4 GPU` (or better). The training cell
   uses CUDA AMP automatically when available.
2. **Ed post** - backend must have `transformers` **and** `sentencepiece` installed (the v3 slow
   tokenizer needs `sentencepiece`; both are confirmed available on the eval backend).
3. **Data** - in Colab, upload `url_with_headlines.csv` when prompted. Locally, the notebook also
   looks for `project-resources/Newsheadlines/url_with_headlines.csv` relative to the repo.

Total runtime on a T4 with AMP: roughly 6-10 min (smaller dataset → fewer steps per epoch).
R-Drop costs 2 forwards/step; with `USE_FGM=True` we add one adversarial forward
(3 forwards/step total).


## 1. Install dependencies

In [ ]:
# transformers + sentencepiece (the v3 slow tokenizer needs it; the eval env has it pre-installed).
!pip -q install "transformers==4.46.0" "sentencepiece>=0.1.99"


## 2. Imports & runtime check

In [ ]:
import os, json, math, random, shutil, time
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoConfig, AutoModelForSequenceClassification, AutoTokenizer,
    get_cosine_schedule_with_warmup,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  device={DEVICE}')
if DEVICE.type == 'cuda':
    print(' ', torch.cuda.get_device_name(0))
    print(' ', f'VRAM total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 3. Configuration

Single seed, no ensemble. Hyperparameters mirror the v3.2 recipe (R-Drop + FGM + LLRD).
Set `DATA_CSV` in the environment to override the default path.


In [ ]:
# ---- Data ----
DATA_CSV     = os.environ.get('DATA_CSV', '/content/url_with_headlines.csv')
TEXT_COL     = 'text'
LABEL_COL    = 'label'
VAL_FRAC     = 0.15
SPLIT_SEED   = 42                       # train/val partition seed
TRAIN_SEED   = 42                       # model init / dropout / shuffle seed

# ---- Model ----
MODEL_NAME   = 'microsoft/deberta-v3-base'
NUM_LABELS   = 2
MAX_LENGTH   = 96

# ---- Training ----
BATCH_SIZE   = 16
GRAD_ACCUM   = 1
EPOCHS       = 12
EARLY_STOP_PATIENCE = 3
MIN_DELTA    = 1e-4
BASE_LR      = 2e-5
LLRD_DECAY   = 0.95
WEIGHT_DECAY = 0.02                     # bumped from 0.01 for v3 (more capacity → more reg)
WARMUP_RATIO = 0.1
LABEL_SMOOTH = 0.1
R_DROP_ALPHA = 1.2                      # bumped from 0.7 for v3 (more capacity → more reg)
AEDA_PROB    = 0.5
AEDA_RATIO   = 0.10
MAX_GRAD_NORM = 1.0
USE_AMP      = True                     # enabled only when DEVICE is CUDA
USE_FGM      = True                     # FGM adversarial step on word_embeddings (one extra forward / step)
FGM_EPSILON  = 1.0                      # perturbation magnitude in unit-direction of grad
NUM_WORKERS  = 0                        # slow tokenizer + notebooks are safest single-process
PIN_MEMORY   = DEVICE.type == 'cuda'

# ---- Paths ----
WORK_DIR     = Path('/content/work') if Path('/content').exists() else Path.cwd() / 'work'
OUT_DIR      = WORK_DIR / 'out'
SUBMIT_DIR   = Path('/content/submission') if Path('/content').exists() else Path.cwd() / 'submission'
for d in (WORK_DIR, OUT_DIR, SUBMIT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Workdir   :', WORK_DIR)
print('Submission:', SUBMIT_DIR)
print('DATA_CSV  :', DATA_CSV)


## 4. Locate or upload `url_with_headlines.csv`

**Colab:** run the cell below, click *Choose Files*, and pick `url_with_headlines.csv` from
`project-resources/Newsheadlines/`.

**Local:** the cell searches the repo first. You can also set `DATA_CSV=/path/to/file.csv` before
launching Jupyter.


In [ ]:
data_path = Path(DATA_CSV).expanduser()
if not data_path.exists():
    candidates = [
        Path.cwd() / 'project-resources/Newsheadlines/url_with_headlines.csv',
        Path.cwd().parent / 'project-resources/Newsheadlines/url_with_headlines.csv',
        Path.cwd().parent.parent / 'project-resources/Newsheadlines/url_with_headlines.csv',
        Path('/content/url_with_headlines.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            data_path = candidate
            break

if not data_path.exists():
    try:
        from google.colab import files
        print('Upload url_with_headlines.csv ...')
        up = files.upload()
        fn = next(iter(up))
        uploaded = Path(fn)
        data_path.parent.mkdir(parents=True, exist_ok=True)
        if uploaded.resolve() != data_path.resolve():
            shutil.move(str(uploaded), str(data_path))
    except ModuleNotFoundError as exc:
        raise FileNotFoundError(
            'Could not find url_with_headlines.csv. Set DATA_CSV or place it under '
            'project-resources/Newsheadlines/.'
        ) from exc

DATA_CSV = str(data_path)
print('CSV at:', DATA_CSV, '(', data_path.stat().st_size//1024, 'KB )')


In [ ]:
# Option B — Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/path/to/url_with_headlines.csv' /content/url_with_headlines.csv


## 5. Load, clean, and inspect backend-style data

In [ ]:
import html
import numbers
import re
import unicodedata
from urllib.parse import urlparse

HEADLINE_COLS = (
    'headline', 'headline_clean', 'scraped_headline',
    'alternative_headline', 'title', 'text',
)
URL_COLS = ('url', 'link', 'URL', 'article_url')
LABEL_COLS = ('label', 'source', 'publisher', 'y')

def find_col(frame, candidates):
    lower = {c.lower(): c for c in frame.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

def label_from_url(url):
    if not isinstance(url, str) or not url:
        return None
    u = url.lower()
    if 'foxnews.com' in u or 'foxbusiness.com' in u:
        return 1
    if 'nbcnews.com' in u or 'today.com' in u or 'msnbc.com' in u:
        return 0
    try:
        host = urlparse(url).netloc.lower()
    except Exception:
        return None
    if 'fox' in host:
        return 1
    if 'nbc' in host:
        return 0
    return None

def label_from_string(v):
    if isinstance(v, str):
        s = v.strip().lower()
        if s in {'1', 'fox', 'foxnews', 'fox news', 'fox_news'}:
            return 1
        if s in {'0', 'nbc', 'nbcnews', 'nbc news', 'nbc_news'}:
            return 0
        if 'fox' in s:
            return 1
        if 'nbc' in s:
            return 0
    elif isinstance(v, numbers.Number) and not pd.isna(v):
        fv = float(v)
        if fv in (0.0, 1.0):
            return int(fv)
    return None

_RE_HTML = re.compile(r'<[^>]+>')
_RE_FOX_SUFFIX = re.compile(r'\s*[\|\-–—]\s*Fox News.*$', re.IGNORECASE)
_RE_NBC_SUFFIX = re.compile(r'\s*[\|\-–—]\s*NBC News.*$', re.IGNORECASE)
_RE_FOX_PREFIX = re.compile(
    r'^\s*(?:FOX\s+NEWS\s+(?:ALERT|EXCLUSIVE|POLL|POWER\s+RANKINGS)'
    r'|Fox\s+News(?:\s+(?:Exclusive|Poll|Power\s+Rankings))?)\s*:\s*',
    re.IGNORECASE,
)
_RE_NBC_PREFIX = re.compile(r'^\s*NBC\s+News\s*:\s*', re.IGNORECASE)
_RE_FOX_NATION = re.compile(r'\bFOX\s+Nation\b', re.IGNORECASE)
_RE_WS = re.compile(r'\s+')

def clean_text(t):
    if not isinstance(t, str):
        return ''
    t = html.unescape(unicodedata.normalize('NFKC', t))
    t = _RE_HTML.sub(' ', t)
    t = _RE_FOX_SUFFIX.sub('', t)
    t = _RE_NBC_SUFFIX.sub('', t)
    t = _RE_FOX_PREFIX.sub('', t)
    t = _RE_NBC_PREFIX.sub('', t)
    t = _RE_FOX_NATION.sub('Nation', t)
    return _RE_WS.sub(' ', t).strip()

raw_df = pd.read_csv(DATA_CSV)
print(f'raw rows: {len(raw_df)}')
print('raw columns:', list(raw_df.columns))

headline_col = find_col(raw_df, HEADLINE_COLS)
url_col = find_col(raw_df, URL_COLS)
label_col = find_col(raw_df, LABEL_COLS)
assert headline_col is not None, f'missing headline column; got {list(raw_df.columns)}'
assert url_col is not None or label_col is not None, 'need URL or label/source column to derive labels'

records = []
for _, row in raw_df.iterrows():
    text = clean_text(row[headline_col])
    if not text:
        continue
    label = label_from_url(row[url_col]) if url_col is not None else None
    if label is None and label_col is not None:
        label = label_from_string(row[label_col])
    if label is None:
        continue
    records.append((text, int(label)))

df = pd.DataFrame(records, columns=[TEXT_COL, LABEL_COL])
_before_dedup = len(df)
conflict_mask = df.groupby(TEXT_COL)[LABEL_COL].transform('nunique') > 1
n_conflict_rows = int(conflict_mask.sum())
n_conflict_texts = int(df.loc[conflict_mask, TEXT_COL].nunique())
if n_conflict_rows:
    print(f'dropping {n_conflict_rows} rows from {n_conflict_texts} ambiguous duplicate headlines')
    df = df.loc[~conflict_mask].copy()
df = df.drop_duplicates(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)
assert set(df[LABEL_COL].unique()) <= {0, 1}, f'unexpected labels: {sorted(df[LABEL_COL].unique())}'
assert df[LABEL_COL].nunique() == 2, 'need both Fox and NBC examples after cleaning'
print(f'dedup: {_before_dedup} -> {len(df)} rows ({_before_dedup - len(df)} dropped)')
print(f'usable rows: {len(df)}')
print(f'text source: {headline_col!r}; label source: {url_col or label_col!r}')
print('class balance (1=Fox / 0=NBC):')
print(df[LABEL_COL].value_counts(normalize=True).round(3).to_string())
df.head(3)


## 6. Stratified train / val split

`SPLIT_SEED=42` is fixed — every run sees the same backend-style partition, so val numbers are
comparable across notebook iterations.


In [ ]:
X_all = df[TEXT_COL].tolist()
y_all = df[LABEL_COL].tolist()
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=VAL_FRAC, random_state=SPLIT_SEED, stratify=y_all,
)
print(f'train={len(X_tr)}  val={len(X_val)}')
print(f'  train Fox rate: {np.mean(y_tr):.3f}')
print(f'  val   Fox rate: {np.mean(y_val):.3f}')


## 7. Tokenizer

DeBERTa-v3 ships a SentencePiece vocab; the slow tokenizer (`DebertaV2Tokenizer`) loads it via
the `sentencepiece` package, which is installed in the cell above and pre-installed on the eval
backend. We stick with `use_fast=False` to keep the runtime path identical between training,
the smoke test, and the evaluator.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
print('tokenizer:', type(tokenizer).__name__)
print('vocab size:', tokenizer.vocab_size)


## 8. `HeadlineDataset` with AEDA augmentation

AEDA = randomly insert punctuation tokens (no synonym swap, no back-translation — those would
erase partisan-style signal). Train fold only.


In [ ]:
PUNCT = ('.', ',', '!', '?', ';', ':')

def aeda_augment(text: str, p_apply=AEDA_PROB, ratio=AEDA_RATIO) -> str:
    if random.random() > p_apply or not text:
        return text
    words = text.split()
    if not words:
        return text
    n_insert = max(1, int(len(words) * ratio))
    for _ in range(n_insert):
        pos = random.randint(0, len(words))
        words.insert(pos, random.choice(PUNCT))
    return ' '.join(words)

class HeadlineDataset(Dataset):
    def __init__(self, texts, labels, augment=False):
        self.texts  = list(texts)
        self.labels = list(labels)
        self.augment = augment
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        t = self.texts[idx]
        if self.augment:
            t = aeda_augment(t)
        return t, int(self.labels[idx])

def collate(batch):
    texts, labels = zip(*batch)
    enc = tokenizer(list(texts), padding=True, truncation=True,
                    max_length=MAX_LENGTH, return_tensors='pt')
    enc['labels'] = torch.tensor(labels, dtype=torch.long)
    return enc


## 9. Training utilities

- `set_seed` — global determinism.
- `rdrop_loss` — symmetric KL on two forward passes + smoothed CE.
- `build_llrd_param_groups` — layer-wise LR decay (deeper = lower LR), classifier head gets
  ~2.6× base LR.
- `FGM` — fast gradient method on `word_embeddings`. After the normal R-Drop backward,
  perturbs the embedding in the unit direction of its grad (magnitude `FGM_EPSILON`),
  runs one extra forward + CE backward to accumulate adversarial gradients, then restores.


In [ ]:
def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def rdrop_loss(logits1, logits2, labels, ce_loss_fn, alpha=R_DROP_ALPHA):
    ce = (ce_loss_fn(logits1, labels) + ce_loss_fn(logits2, labels)) / 2
    p1 = F.log_softmax(logits1, dim=-1); p2 = F.log_softmax(logits2, dim=-1)
    q1 = F.softmax(logits1, dim=-1);     q2 = F.softmax(logits2, dim=-1)
    kl = (F.kl_div(p1, q2, reduction='batchmean')
          + F.kl_div(p2, q1, reduction='batchmean')) / 2
    return ce + alpha * kl

def build_llrd_param_groups(model, lr=BASE_LR, decay=LLRD_DECAY, weight_decay=WEIGHT_DECAY):
    n_layers = model.config.num_hidden_layers
    no_decay = ('bias', 'LayerNorm.weight', 'LayerNorm.bias')
    named = list(model.named_parameters())
    groups = []
    seen = set()

    def add_group(params, group_lr, group_wd):
        params = [p for p in params if id(p) not in seen]
        if not params:
            return
        seen.update(id(p) for p in params)
        groups.append({'params': params, 'lr': group_lr, 'weight_decay': group_wd})

    embed_lr = lr * (decay ** (n_layers + 1))
    embed_decay = [p for n,p in named if n.startswith('deberta.embeddings.') and not any(nd in n for nd in no_decay)]
    embed_nodec = [p for n,p in named if n.startswith('deberta.embeddings.') and     any(nd in n for nd in no_decay)]
    add_group(embed_decay, embed_lr, weight_decay)
    add_group(embed_nodec, embed_lr, 0.0)
    for i in range(n_layers):
        layer_lr = lr * (decay ** (n_layers - i))
        prefix = f'deberta.encoder.layer.{i}.'
        d = [p for n,p in named if n.startswith(prefix) and not any(nd in n for nd in no_decay)]
        n_ = [p for n,p in named if n.startswith(prefix) and     any(nd in n for nd in no_decay)]
        add_group(d, layer_lr, weight_decay)
        add_group(n_, layer_lr, 0.0)
    head_lr = lr * 2.6
    hd = [p for n,p in named if not n.startswith('deberta.') and not any(nd in n for nd in no_decay)]
    hn = [p for n,p in named if not n.startswith('deberta.') and     any(nd in n for nd in no_decay)]
    add_group(hd, head_lr, weight_decay)
    add_group(hn, head_lr, 0.0)

    leftovers = [(n, p) for n, p in named if p.requires_grad and id(p) not in seen]
    if leftovers:
        print(f'LLRD fallback group for {len(leftovers)} tensor(s): {[n for n, _ in leftovers[:5]]}')
    leftover_decay = [p for n, p in leftovers if not any(nd in n for nd in no_decay)]
    leftover_nodec = [p for n, p in leftovers if any(nd in n for nd in no_decay)]
    add_group(leftover_decay, lr, weight_decay)
    add_group(leftover_nodec, lr, 0.0)

    n_params = sum(1 for _, p in named if p.requires_grad)
    grouped = sum(len(g['params']) for g in groups)
    assert grouped == n_params, f'optimizer grouped {grouped}/{n_params} trainable tensors'
    return groups


class FGM:
    """Fast Gradient Method adversarial perturbation on a named embedding parameter.

    Usage in a training step (after the normal backward pass has populated grads):
        fgm.attack()
        loss_adv = ce(model(inputs).logits, labels)
        loss_adv.backward()      # accumulates onto existing grads
        fgm.restore()
    """
    def __init__(self, model, epsilon=1.0, emb_name='word_embeddings'):
        self.model = model
        self.epsilon = epsilon
        self.emb_name = emb_name
        self.backup = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if not param.requires_grad or self.emb_name not in name or param.grad is None:
                continue
            self.backup[name] = param.data.clone()
            norm = torch.norm(param.grad.detach())
            if norm.item() > 0 and not torch.isnan(norm) and not torch.isinf(norm):
                r_at = self.epsilon * param.grad / norm
                param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}


## 10. Train

Best checkpoint selected on **val_f1**. Saves directly to `submission/model.pt` (no re-keying
required — the HF model's state_dict keys match `model.py`'s `self.model` exactly).


In [ ]:
amp_enabled = USE_AMP and DEVICE.type == 'cuda'
if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
else:
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

def amp_context():
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast('cuda', enabled=amp_enabled)
    return torch.cuda.amp.autocast(enabled=amp_enabled)

def move_batch_to_device(batch):
    labels = batch.pop('labels').to(DEVICE, non_blocking=PIN_MEMORY)
    batch = {k: v.to(DEVICE, non_blocking=PIN_MEMORY) for k, v in batch.items()}
    return batch, labels

def evaluate(model, loader):
    model.eval()
    all_p, all_t = [], []
    with torch.inference_mode():
        for batch in loader:
            batch, labels = move_batch_to_device(batch)
            with amp_context():
                logits = model(**batch).logits
            all_p.extend(logits.argmax(-1).cpu().tolist())
            all_t.extend(labels.cpu().tolist())
    return accuracy_score(all_t, all_p), f1_score(all_t, all_p, average='binary')

set_seed(TRAIN_SEED)

train_ds = HeadlineDataset(X_tr,  y_tr,  augment=True)
val_ds   = HeadlineDataset(X_val, y_val, augment=False)
g = torch.Generator(); g.manual_seed(TRAIN_SEED)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
    generator=g, drop_last=False, num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE*2, shuffle=False, collate_fn=collate,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
)

config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS,
                                     problem_type='single_label_classification')
model  = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config)
model.to(DEVICE)

optim = torch.optim.AdamW(build_llrd_param_groups(model))
total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup      = int(total_steps * WARMUP_RATIO)
sched = get_cosine_schedule_with_warmup(optim, warmup, total_steps)
ce_loss = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

fgm = FGM(model, epsilon=FGM_EPSILON) if USE_FGM else None

history, best_f1, best_epoch, no_improve = [], -1.0, -1, 0
best_path = SUBMIT_DIR / 'model.pt'
print(f'AMP enabled: {amp_enabled}  |  FGM enabled: {USE_FGM}')

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0, run_loss, n_seen = time.time(), 0.0, 0
    optim.zero_grad(set_to_none=True)
    for step, batch in enumerate(train_loader):
        batch, labels = move_batch_to_device(batch)

        # Normal R-Drop pass (2 forwards w/ different dropout masks)
        with amp_context():
            logits1 = model(**batch).logits
            logits2 = model(**batch).logits
            loss = rdrop_loss(logits1, logits2, labels, ce_loss) / GRAD_ACCUM
        scaler.scale(loss).backward()

        # Adversarial pass: perturb word_embeddings, single forward + CE backward
        if fgm is not None:
            fgm.attack()
            with amp_context():
                logits_adv = model(**batch).logits
                loss_adv = ce_loss(logits_adv, labels) / GRAD_ACCUM
            scaler.scale(loss_adv).backward()
            fgm.restore()

        run_loss += loss.item() * GRAD_ACCUM * labels.size(0)
        n_seen   += labels.size(0)
        is_update_step = ((step + 1) % GRAD_ACCUM == 0) or ((step + 1) == len(train_loader))
        if is_update_step:
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optim)
            scaler.update()
            sched.step()
            optim.zero_grad(set_to_none=True)
    train_loss = run_loss / max(n_seen, 1)
    val_acc, val_f1 = evaluate(model, val_loader)
    history.append({'epoch': epoch, 'train_loss': train_loss,
                    'val_acc': val_acc, 'val_f1': val_f1,
                    'sec': round(time.time()-t0, 1)})
    print(f'  epoch {epoch:2d}  loss={train_loss:.4f}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}  ({history[-1]["sec"]}s)')
    if val_f1 > best_f1 + MIN_DELTA:
        best_f1, best_epoch, no_improve = val_f1, epoch, 0
        torch.save(model.state_dict(), best_path)
    else:
        no_improve += 1
        if no_improve >= EARLY_STOP_PATIENCE:
            print(f'  early stop @ epoch {epoch} (best epoch {best_epoch}, val_f1={best_f1:.4f})')
            break

pd.DataFrame(history).to_csv(OUT_DIR / 'history.csv', index=False)
summary = {
    'model_name': MODEL_NAME,
    'data_csv': str(DATA_CSV),
    'best_epoch': best_epoch,
    'best_val_f1': best_f1,
    'val_frac': VAL_FRAC,
    'split_seed': SPLIT_SEED,
    'train_seed': TRAIN_SEED,
    'max_length': MAX_LENGTH,
    'amp_enabled': amp_enabled,
    'use_fgm': USE_FGM,
    'fgm_epsilon': FGM_EPSILON,
    'r_drop_alpha': R_DROP_ALPHA,
    'weight_decay': WEIGHT_DECAY,
    'base_lr': BASE_LR,
}
(OUT_DIR / 'training_summary.json').write_text(json.dumps(summary, indent=2))
print(f'\n✓ saved {best_path}  ({best_path.stat().st_size/1e6:.1f} MB)')
print(f'  best val_f1={best_f1:.4f}  @ epoch {best_epoch}')
del model; torch.cuda.empty_cache()


## 11. Write `model.py`

In [ ]:
MODEL_PY = '"""\nmodel.py - Project B: News Source Classification (1=FoxNews, 0=NBC)\n\nSingle `microsoft/deberta-v3-base` classifier. The evaluator instantiates\n`Model(weights_path="__no_weights__.pth")`, loads `model.pt` externally, and\nthen calls `predict(batch)`. A real `weights_path` is also supported for local\nsmoke tests and direct reuse.\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Iterable, List, Mapping, Optional\n\nimport torch\nfrom torch import nn\nfrom transformers import (\n    AutoConfig,\n    AutoModelForSequenceClassification,\n    AutoTokenizer,\n)\n\nMODEL_NAME = "microsoft/deberta-v3-base"\nNUM_LABELS = 2\nMAX_LENGTH = 96\n_SENTINEL_WEIGHTS = {"", "__no_weights__.pth", None}\n\n\ndef _best_device() -> torch.device:\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef _normalize_state_dict(state_dict: Mapping[str, Any]) -> dict[str, Any]:\n    normalized: dict[str, Any] = {}\n    for key, value in state_dict.items():\n        if key.startswith("module."):\n            key = key[len("module.") :]\n        if key.startswith("model."):\n            key = key[len("model.") :]\n        normalized[key] = value\n    return normalized\n\n\ndef _load_matching(target: nn.Module, state_dict: Mapping[str, Any]) -> int:\n    target_state = target.state_dict()\n    filtered = {\n        key: value\n        for key, value in state_dict.items()\n        if key in target_state\n        and isinstance(value, torch.Tensor)\n        and target_state[key].shape == value.shape\n    }\n    if filtered:\n        target.load_state_dict(filtered, strict=False)\n    return len(filtered)\n\n\nclass Model(nn.Module):\n    def __init__(self, weights_path: Optional[str] = None) -> None:\n        super().__init__()\n        self.config = AutoConfig.from_pretrained(\n            MODEL_NAME,\n            num_labels=NUM_LABELS,\n            problem_type="single_label_classification",\n        )\n        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)\n        self.model = AutoModelForSequenceClassification.from_config(self.config)\n        self.device = _best_device()\n        self.to(self.device)\n        self._maybe_load_weights(weights_path)\n        self.eval()\n\n    def _maybe_load_weights(self, weights_path: Optional[str]) -> None:\n        if weights_path in _SENTINEL_WEIGHTS:\n            return\n        path = Path(str(weights_path)).expanduser()\n        if not path.exists():\n            return\n        checkpoint = torch.load(path, map_location="cpu")\n        raw_state = (\n            checkpoint.get("state_dict", checkpoint)\n            if isinstance(checkpoint, dict)\n            else checkpoint\n        )\n        if not isinstance(raw_state, Mapping):\n            raise RuntimeError(\n                "Checkpoint must be a state_dict or a dict with a state_dict key."\n            )\n        state_dict = _normalize_state_dict(raw_state)\n        loaded = _load_matching(self.model, state_dict) + _load_matching(\n            self, state_dict\n        )\n        if loaded == 0:\n            raise RuntimeError("No checkpoint tensors matched the DeBERTa classifier.")\n        self.to(self.device)\n\n    def eval(self) -> "Model":\n        super().eval()\n        self.model.eval()\n        return self\n\n    def _encode(self, batch: Iterable[Any]) -> dict[str, torch.Tensor]:\n        texts = [str(x) if x is not None else "" for x in batch]\n        return self.tokenizer(\n            texts,\n            padding=True,\n            truncation=True,\n            max_length=MAX_LENGTH,\n            return_tensors="pt",\n        )\n\n    @torch.no_grad()\n    def predict(self, batch: Iterable[Any]) -> List[int]:\n        texts = list(batch)\n        if not texts:\n            return []\n        enc = self._encode(texts)\n        enc = {k: v.to(self.device, non_blocking=True) for k, v in enc.items()}\n        logits = self.model(**enc).logits\n        return logits.argmax(dim=-1).detach().cpu().tolist()\n\n    def forward(self, batch: Iterable[Any]) -> torch.Tensor:\n        enc = self._encode(batch)\n        enc = {k: v.to(self.device, non_blocking=True) for k, v in enc.items()}\n        return self.model(**enc).logits\n\n\ndef get_model() -> Model:\n    return Model()\n'
(SUBMIT_DIR / 'model.py').write_text(MODEL_PY)
print('✓ wrote', SUBMIT_DIR / 'model.py', f"({(SUBMIT_DIR / 'model.py').stat().st_size} bytes)")


## 12. Write `preprocess.py`

Same logic as the root `preprocess.py`: read CSV → clean headline → derive label from URL host.
**Fox=1, NBC=0**, matching how we trained.


In [ ]:
PREPROCESS_PY = '"""\npreprocess.py - Project B: News Source Classification (Fox News vs NBC News)\n\nContract (from eval_project_b.py):\n    prepare_data(csv_path: str) -> (X, y)\n        X: list of cleaned headline strings\n        y: list[int], with FoxNews=1 and NBC=0\n"""\n\nfrom __future__ import annotations\n\nimport html\nimport numbers\nimport re\nimport unicodedata\nfrom typing import List, Optional, Sequence, Tuple\nfrom urllib.parse import urlparse\n\nimport pandas as pd\n\n\nLABEL_FOX = 1\nLABEL_NBC = 0\nLABEL_NAMES = {LABEL_FOX: "FoxNews", LABEL_NBC: "NBC"}\n\n_HEADLINE_COLS: Sequence[str] = (\n    "headline",\n    "headline_clean",\n    "scraped_headline",\n    "alternative_headline",\n    "title",\n    "text",\n)\n_URL_COLS: Sequence[str] = ("url", "link", "URL", "article_url")\n_LABEL_COLS: Sequence[str] = ("label", "source", "publisher", "y")\n\n\ndef _find_col(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:\n    lower = {c.lower(): c for c in df.columns}\n    for c in candidates:\n        if c.lower() in lower:\n            return lower[c.lower()]\n    return None\n\n\ndef _label_from_url(url: object) -> Optional[int]:\n    if not isinstance(url, str) or not url:\n        return None\n    u = url.lower()\n    if "foxnews.com" in u or "foxbusiness.com" in u:\n        return LABEL_FOX\n    if "nbcnews.com" in u or "today.com" in u or "msnbc.com" in u:\n        return LABEL_NBC\n    try:\n        host = urlparse(url).netloc.lower()\n    except Exception:\n        return None\n    if "fox" in host:\n        return LABEL_FOX\n    if "nbc" in host:\n        return LABEL_NBC\n    return None\n\n\ndef _label_from_string(v: object) -> Optional[int]:\n    if isinstance(v, str):\n        s = v.strip().lower()\n        if s in {"1", "fox", "foxnews", "fox news", "fox_news"}:\n            return LABEL_FOX\n        if s in {"0", "nbc", "nbcnews", "nbc news", "nbc_news"}:\n            return LABEL_NBC\n        if "fox" in s:\n            return LABEL_FOX\n        if "nbc" in s:\n            return LABEL_NBC\n    elif isinstance(v, numbers.Number) and not pd.isna(v):\n        fv = float(v)\n        if fv in (0.0, 1.0):\n            return int(fv)\n    return None\n\n\n_RE_HTML = re.compile(r"<[^>]+>")\n_RE_FOX_SUFFIX = re.compile(r"\\s*[\\|\\-–—]\\s*Fox News.*$", re.IGNORECASE)\n_RE_NBC_SUFFIX = re.compile(r"\\s*[\\|\\-–—]\\s*NBC News.*$", re.IGNORECASE)\n_RE_FOX_PREFIX = re.compile(\n    r"^\\s*(?:FOX\\s+NEWS\\s+(?:ALERT|EXCLUSIVE|POLL|POWER\\s+RANKINGS)"\n    r"|Fox\\s+News(?:\\s+(?:Exclusive|Poll|Power\\s+Rankings))?)\\s*:\\s*",\n    re.IGNORECASE,\n)\n_RE_NBC_PREFIX = re.compile(r"^\\s*NBC\\s+News\\s*:\\s*", re.IGNORECASE)\n_RE_FOX_NATION = re.compile(r"\\bFOX\\s+Nation\\b", re.IGNORECASE)\n_RE_WS = re.compile(r"\\s+")\n\n\ndef _clean_text(t: object) -> str:\n    if not isinstance(t, str):\n        return ""\n    t = html.unescape(unicodedata.normalize("NFKC", t))\n    t = _RE_HTML.sub(" ", t)\n    t = _RE_FOX_SUFFIX.sub("", t)\n    t = _RE_NBC_SUFFIX.sub("", t)\n    t = _RE_FOX_PREFIX.sub("", t)\n    t = _RE_NBC_PREFIX.sub("", t)\n    t = _RE_FOX_NATION.sub("Nation", t)\n    return _RE_WS.sub(" ", t).strip()\n\n\ndef prepare_data(path: str) -> Tuple[List[str], List[int]]:\n    df = pd.read_csv(path)\n\n    headline_col = _find_col(df, _HEADLINE_COLS)\n    url_col = _find_col(df, _URL_COLS)\n    label_col = _find_col(df, _LABEL_COLS)\n\n    if headline_col is None:\n        raise ValueError(\n            f"No headline column found. Looked for {list(_HEADLINE_COLS)}; "\n            f"got columns {list(df.columns)}"\n        )\n    if url_col is None and label_col is None:\n        raise ValueError(\n            f"Need a URL column ({list(_URL_COLS)}) or label column "\n            f"({list(_LABEL_COLS)}) to derive y; got {list(df.columns)}"\n        )\n\n    X: List[str] = []\n    y: List[int] = []\n    for _, row in df.iterrows():\n        text = _clean_text(row[headline_col])\n        if not text:\n            continue\n\n        label: Optional[int] = None\n        if url_col is not None:\n            label = _label_from_url(row[url_col])\n        if label is None and label_col is not None:\n            label = _label_from_string(row[label_col])\n        if label is None:\n            continue\n\n        X.append(text)\n        y.append(label)\n\n    return X, y\n\n\nif __name__ == "__main__":\n    import sys\n\n    csv_path = (\n        sys.argv[1]\n        if len(sys.argv) > 1\n        else "project-resources/Newsheadlines/url_with_headlines.csv"\n    )\n    X, y = prepare_data(csv_path)\n    fox = sum(1 for v in y if v == LABEL_FOX)\n    nbc = sum(1 for v in y if v == LABEL_NBC)\n    print(f"n={len(X)}  FoxNews={fox}  NBC={nbc}")\n    for i in range(min(3, len(X))):\n        print(f"[{LABEL_NAMES[y[i]]}] {X[i]}")\n'
(SUBMIT_DIR / 'preprocess.py').write_text(PREPROCESS_PY)
print('✓ wrote', SUBMIT_DIR / 'preprocess.py', f"({(SUBMIT_DIR / 'preprocess.py').stat().st_size} bytes)")


## 13. Smoke test — load submission artifacts and predict

Reproduces what the eval harness does: import `model.py`, instantiate `Model()`, load `model.pt`
via the harness's robust `load_state_dict`, run `predict()` on the held-out split, and report
accuracy. This is a held-out check on the same text representation that the backend will pass
to `predict()`.


In [ ]:
import sys, importlib
if str(SUBMIT_DIR) not in sys.path:
    sys.path.insert(0, str(SUBMIT_DIR))
for mod in ('model', 'preprocess'):
    if mod in sys.modules: del sys.modules[mod]
import model as _smoke_model
import preprocess as _smoke_preproc

def _normalize(k):
    if k.startswith('module.'): k = k[len('module.'):]
    if k.startswith('model.'):  k = k[len('model.'):]
    return k
def _load_into(target, sd):
    if target is None: return 0
    tgt = target.state_dict()
    f = {k: v for k, v in sd.items() if k in tgt and isinstance(v, torch.Tensor) and tgt[k].shape == v.shape}
    if not f: return 0
    target.load_state_dict(f, strict=False)
    return len(f)

smoke = _smoke_model.Model(weights_path='__no_weights__.pth')
ckpt = torch.load(SUBMIT_DIR / 'model.pt', map_location='cpu')
sd_raw = ckpt['state_dict'] if isinstance(ckpt, dict) and 'state_dict' in ckpt else ckpt
sd = {_normalize(k): v for k, v in sd_raw.items()}
loaded = _load_into(getattr(smoke, 'model', None), sd) + _load_into(smoke, sd)
print(f'loaded {loaded} tensors from model.pt')
assert loaded > 0, 'no tensors matched — check key prefixes'
smoke.eval()

preds = []
BS = 32
for i in range(0, len(X_val), BS):
    preds.extend(smoke.predict(X_val[i:i+BS]))
val_acc = accuracy_score(y_val, preds)
val_f1  = f1_score(y_val, preds, average='binary')
print(f'\n smoke val_acc={val_acc:.4f}  val_f1={val_f1:.4f}')
print('\n class report:')
print(classification_report(y_val, preds, target_names=['NBC(0)','Fox(1)']))


## 14. Package `submission.zip` and download

In [ ]:
zip_path = '/content/submission.zip'
if os.path.exists(zip_path): os.remove(zip_path)
shutil.make_archive('/content/submission', 'zip', root_dir=str(SUBMIT_DIR))
print(f'✓ {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('\nContents:')
for p in sorted(SUBMIT_DIR.iterdir()):
    print(f'  {p.name:20s}  {p.stat().st_size/1e6:8.2f} MB')

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print('skip download:', e)
